# 蜃景 × lightx2v —— 纯 t2v 文生视频部署（无 ComfyUI）

为「纯 t2v + lightx2v、彻底不用 ComfyUI」准备。lightx2v 是你那套 4 步蒸馏 LoRA 的娘家、专做 Wan 快速推理。
**本笔记本不装 ComfyUI、不出图、不锁脸、不换脸**——只做：文本→视频(lightx2v) + 配音 + 字幕。

**⚠️ 关键未验证项（先跑 §3/§4/§5 确认，再往后）：** lightx2v 在 Blackwell(sm_120) 上能否装上
（尤其注意力算子）、server 能否起来。装不上时优先走它的 **nvfp4** 路线（不依赖 SageAttention）。
带 ★ 的命令是按 lightx2v 文档写的**脚手架**，以你的 lightx2v README/scripts 实际为准、按需改。

**用法**：运行时选 GPU → 逐格 Run。角色 LoRA 训练仍在 `colab_deploy.ipynb` 的 LW1/LW2（用 ai-toolkit，与本部署独立）。

In [ ]:
# === §1 参数 + Drive + GPU 探测（设 T2V_PRECISION）===
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'
BRANCH   = 'main'
import os, torch as _t
os.makedirs('/content', exist_ok=True)
if _t.cuda.is_available():
    cc = _t.cuda.get_device_capability(0); name = _t.cuda.get_device_name(0)
    vram = _t.cuda.get_device_properties(0).total_memory / (1024**3)
    native_fp8 = (cc[0] > 8) or (cc[0] == 8 and cc[1] >= 9)
    # 原生 fp8(Blackwell/Hopper/Ada)→ lightx2v 走 fp8/nvfp4 最快;A100(sm80)无原生 fp8 → int8/bf16
    _prec = 'fp8' if native_fp8 else 'fp16'
    os.environ['T2V_PRECISION'] = _prec
    print(f'🖥 GPU: {name} | sm{cc[0]}.{cc[1]} | {vram:.0f}G | 原生FP8={native_fp8} → T2V_PRECISION={_prec}')
    if cc[0] >= 12:
        print('   ★Blackwell(sm120):lightx2v 注意力算子可能要预编 wheel 或走 nvfp4;§3 留意报错。')
else:
    os.environ['T2V_PRECISION'] = 'fp16'
    print('⚠️ 没检测到 GPU！代码执行程序 → 更改运行时类型 → 选 GPU')
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'; CACHE=DRIVE+'/mirage_models'; TOOLS=DRIVE+'/mirage_tools'
PIP_CACHE=TOOLS+'/pip_cache'
for p in (CACHE, TOOLS, PIP_CACHE): os.makedirs(p, exist_ok=True)
os.environ['PIP_CACHE_DIR']=PIP_CACHE; os.environ['HF_HOME']=TOOLS+'/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('环境就绪 | 模型:', CACHE)

In [ ]:
# === §2 拉取/更新蜃景仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)  # 子进程继承内核 torch(sm120 必需)
print('仓库就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（★sm_120 上的关键关：注意力算子装不上就走 nvfp4）===
# 核心框架:pip 装。注意力算子(SageAttention/FlashAttention)按 lightx2v 文档单独装——这步是 Blackwell 风险点。
!pip -q install -v "git+https://github.com/ModelTC/lightx2v.git" || echo '⚠️ lightx2v 主框架装失败,看上面报错'
# ★注意力算子(以 lightx2v 文档为准;Blackwell 优先找预编 wheel 或 nvfp4 路线):
# !pip -q install sageattention   # 或 flash-attn,sm120 多半要对应 torch 的预编 wheel
import importlib.util as _u
print('lightx2v 可导入:' , bool(_u.find_spec('lightx2v')))
print('sageattention 可用:', bool(_u.find_spec('sageattention')), '(没有就走 nvfp4,见 lightx2v 文档)')

In [ ]:
# === §4 下 lightx2v 的 Wan2.2-T2V 蒸馏权重（★仓库/文件名以 lightx2v HF 为准）===
# lightx2v 直接发预量化的蒸馏权重(bf16/fp8/nvfp4),省得自己挂蒸馏 LoRA。按 T2V_PRECISION 选档。
import os
LX_MODELS = '/content/drive/MyDrive/mirage_models/lightx2v_t2v'   # 持久化 Drive
os.makedirs(LX_MODELS, exist_ok=True)
# ★以下仓库名/子路径以 https://huggingface.co/lightx2v 实际为准(Wan2.2-Distill-Models 系列)
!hf download lightx2v/Wan2.2-Distill-Models --local-dir {LX_MODELS} || echo '⚠️ 权重没下到:去 huggingface.co/lightx2v 核对确切仓库名/文件,改这行'
os.environ['LIGHTX2V_MODEL_T2V'] = LX_MODELS
print('lightx2v t2v 权重目录:', LX_MODELS, '| 精度档:', os.environ.get('T2V_PRECISION'))

In [ ]:
# === §5 起 lightx2v HTTP server（★启动命令以 lightx2v scripts/server 为准；端口 8189）===
# lightx2v 自带 FastAPI server。clone 其仓库拿 scripts/server,后台起服务(脱离内核,停 cell 不杀)。
import os, sys; sys.path.insert(0, '/content/mirage/colab'); import persist
if not os.path.isdir('/content/LightX2V/.git'):
    !git clone https://github.com/ModelTC/lightx2v /content/LightX2V
# ★真实启动方式以 /content/LightX2V/scripts/server/ 与其 README 为准。下面是脚手架占位:
#   通常需要一个 config(指定 model_cls=wan2.2 / task=t2v / model_path=$LIGHTX2V_MODEL_T2V / port=8189)。
os.environ['LIGHTX2V_PORT'] = '8189'
print('★请按 lightx2v README 在此起 server,例如:')
print('   cd /content/LightX2V && bash scripts/server/start_server.sh   # 配 model_path/port/task=t2v')
print('   起好后健康检查:  curl http://127.0.0.1:8189/v1/service/status')
# 起好后填:
os.environ['LIGHTX2V_BASE_URL'] = 'http://127.0.0.1:8189'
# persist.start_bg([...lightx2v server 启动命令...], '/content/lightx2v.log')   # ← 核对命令后启用
print('LIGHTX2V_BASE_URL =', os.environ['LIGHTX2V_BASE_URL'])

In [ ]:
# === §6 装蜃景后端依赖 + 写 .env（不配 ComfyUI → 出图/换脸不注册；t2v 走 lightx2v）===
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
import os
lines = [
    'COMFYUI_BASE_URL=',                                  # ★空 = 不起/不连 ComfyUI(纯 t2v)
    'LIGHTX2V_ENABLED=true',
    'LIGHTX2V_BASE_URL=' + os.environ.get('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
    'LIGHTX2V_MODEL_T2V=' + os.environ.get('LIGHTX2V_MODEL_T2V', ''),
    'T2V_PROVIDER=lightx2v-t2v',                          # t2v 出片走 lightx2v
    'T2V_PRECISION=' + os.environ.get('T2V_PRECISION', 'fp8'),
    'VIDEO_PROVIDER_DEFAULT=wan2.2',
]
open('.env', 'w').write('\n'.join(lines) + '\n')
print('.env 写好(纯 t2v + lightx2v,无 ComfyUI):')
print('\n'.join('  ' + l for l in lines))

In [ ]:
# === §7 起蜃景后端（读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端',
    ['uvicorn','mirage.main_api:app','--host','0.0.0.0','--port','8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn',
    cwd='/content/mirage', timeout=120)

In [ ]:
# === §8 cloudflared 暴露后端（拿公网地址）===
import sys, re, time, os, subprocess
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m: url = m[-1]; break
print('✅ 公网地址:', url) if url else print('⚠ 60s 未就绪 → !tail -20 /content/cf.log')
print('打开它 → 工作台「出片模式 = t2v」→ 一键全自动出片(走 lightx2v,无 ComfyUI)。配音+字幕自动。')

In [ ]:
# === §9 实时日志（看 lightx2v 出片采样到第几步；tail -f 阻塞，停本格只停 tail、不杀服务）===
# 看到采样进度 % = 正在出片；卡住 / 报错也在这显形。
# 出片走 lightx2v server → 日志在 /content/lightx2v.log（★确保 §5 起 server 时把日志写到这个文件）。
# 后端日志看 /content/api.log；隧道日志看 /content/cf.log。要看哪个把下面文件名换掉即可。
import os
_log = '/content/lightx2v.log'
open(_log, 'a').close()   # 没有就建个空文件，免 tail 报错
!tail -n 80 -f {_log}


---
## 说明
- **本部署纯 t2v**：没有出图/选图/PuLID/换脸（那些要 ComfyUI）。身份靠训好的 Wan-T2V 角色 LoRA。
- **角色 LoRA 训练**：仍用 `colab_deploy.ipynb` 的 **LW1/LW2**（ai-toolkit，与 lightx2v 推理独立）；
  训好的 LoRA 放进 lightx2v 的 lora_configs（后端 `WAN_T2V_LORA_HIGH/LOW` 或项目设置）。
- **配音 + 字幕**：合成时自动（edge-tts 按角色音色 + 烧字幕），与后端无关。
- **★首跑核对清单**：§3 lightx2v + 注意力算子在 Blackwell 装上了吗 → §4 权重仓库名对吗 →
  §5 server 起命令对吗、`/v1/service/status` 通吗 → 出一条片确认 `providers/lightx2v.py` 的请求/取片字段对得上
  （对不上就按真机改 `_extract_output` / payload）。